# Lab: Comprehensive Text Data Cleaning with Python

This lab guides you through a complete **text data cleaning pipeline** in Python: from noise removal through lemmatization.

## Objective

Transform raw, unstructured text into a standardized form suitable for NLP by addressing noise, normalization, tokenization, and linguistic reduction.

## Prerequisites

Install the required libraries (uncomment and run the next cell if needed):

```bash
pip install pandas nltk pyspellchecker emoji
```

In [ ]:
# Uncomment to install dependencies:
# %pip install pandas nltk pyspellchecker emoji

  Using cached emoji-2.15.0-py3-none-any.whl.metadata (5.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 1.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 473.6 kB/s eta 0:00:0000:0100:01
Using cached emoji-2.15.0-py3-none-any.whl (608 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [emoji]32m2/3 [emoji]
Note: you may need to restart the kernel to use updated packages.


---

## Step 1: Initialize the Data

We create a Pandas `DataFrame` with **noisy** text: HTML tags, URLs, spelling errors, and similar issues.

In [1]:
import pandas as pd

data = {
    'raw_text': [
        "<p>Check https://example.com for the 2023 report!</p>",
        "The runner runs quickly thru the running track... 🏃‍♂️",
        "L'ensemble is better; I recieved the package in the U.S.A.",
        "Their coming too sea if its reel.",
        "I love this movie! It's sweet, but with satirical humor."
    ]
}

df = pd.DataFrame(data)
print("Original Data:")
print(df)

Original Data:
                                            raw_text
0  <p>Check https://example.com for the 2023 repo...
1  The runner runs quickly thru the running track...
2  L'ensemble is better; I recieved the package i...
3                  Their coming too sea if its reel.
4  I love this movie! It's sweet, but with satiri...


---

## Step 2: Noise Removal (RegEx & Emojis)

Remove HTML tags, URLs, and normalize numbers; convert emojis to text descriptions; strip special characters while keeping alphanumeric tokens and spaces.

In [5]:
import re
import emoji


def clean_noise(text):
    # 1. Remove HTML tags
    text = re.sub('<.*?>', '', text)
    # 2. Replace URLs with [URL] placeholder
    text = re.sub(r'http\S+|www\S+|https\S+', '[URL]', text)
    # 3. Standardize numbers to [NUM]
    text = re.sub(r'\d+', '[NUM]', text)
    # 4. Convert emojis to text
    text = emoji.demojize(text)
    # 5. Remove special characters (keep alphanumeric and spaces)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text


df['cleaned_noise'] = df['raw_text'].apply(clean_noise)
df

,raw_text,cleaned_noise
0,<p>Check https://example.com for the 2023 repo...,Check URL for the NUM report
1,The runner runs quickly thru the running track...,The runner runs quickly thru the running track...
2,L'ensemble is better; I recieved the package i...,Lensemble is better I recieved the package in ...
3,Their coming too sea if its reel.,Their coming too sea if its reel
4,"I love this movie! It's sweet, but with satiri...",I love this movie Its sweet but with satirical...


---

## Step 3: Text Normalization & Spell Checking

Apply **case folding** (lowercase) to shrink vocabulary size, then use `pyspellchecker` to correct typos such as "recieved".

In [6]:
from spellchecker import SpellChecker

spell = SpellChecker()


def normalize_and_spell(text):
    text = text.lower()
    words = text.split()
    corrected_words = [spell.correction(w) if spell.correction(w) else w for w in words]
    return " ".join(corrected_words)


df['normalized'] = df['cleaned_noise'].apply(normalize_and_spell)
df

,raw_text,cleaned_noise,normalized
0,<p>Check https://example.com for the 2023 repo...,Check URL for the NUM report,check curl for the mum report
1,The runner runs quickly thru the running track...,The runner runs quickly thru the running track...,the runner runs quickly thru the running track...
2,L'ensemble is better; I recieved the package i...,Lensemble is better I recieved the package in ...,ensemble is better i received the package in t...
3,Their coming too sea if its reel.,Their coming too sea if its reel,their coming too sea if its reel
4,"I love this movie! It's sweet, but with satiri...",I love this movie Its sweet but with satirical...,i love this movie its sweet but with satirical...


---

## Step 4: Tokenization & Stop Word Removal

Split text into **tokens** and drop **stop words** that carry little meaning (e.g. "the", "is").

In [7]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)  # NLTK 3.8+ uses punkt_tab for word_tokenize
nltk.download('stopwords', quiet=True)

stop_words = set(stopwords.words('english'))


def tokenize_and_remove_stop(text):
    tokens = word_tokenize(text)
    filtered = [w for w in tokens if w not in stop_words]
    return filtered


df['tokens'] = df['normalized'].apply(tokenize_and_remove_stop)
df

,raw_text,cleaned_noise,normalized,tokens
0,<p>Check https://example.com for the 2023 repo...,Check URL for the NUM report,check curl for the mum report,"[check, curl, mum, report]"
1,The runner runs quickly thru the running track...,The runner runs quickly thru the running track...,the runner runs quickly thru the running track...,"[runner, runs, quickly, thru, running, track, ..."
2,L'ensemble is better; I recieved the package i...,Lensemble is better I recieved the package in ...,ensemble is better i received the package in t...,"[ensemble, better, received, package, us]"
3,Their coming too sea if its reel.,Their coming too sea if its reel,their coming too sea if its reel,"[coming, sea, reel]"
4,"I love this movie! It's sweet, but with satiri...",I love this movie Its sweet but with satirical...,i love this movie its sweet but with satirical...,"[love, movie, sweet, satirical, humor]"


---

## Step 5: Stemming vs. Lemmatization

Compare **stemming** (rule-based affix chopping) and **lemmatization** (dictionary-style base forms).

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()


def apply_stemming(tokens):
    return [stemmer.stem(w) for w in tokens]


def apply_lemmatization(tokens):
    return [lemmatizer.lemmatize(w) for w in tokens]


df['stemmed'] = df['tokens'].apply(apply_stemming)
df['lemmatized'] = df['tokens'].apply(apply_lemmatization)

print("\nFinal Processed Data (Lemmatized):")
print(df[['raw_text', 'lemmatized']])
df


Final Processed Data (Lemmatized):
                                            raw_text  \
0  <p>Check https://example.com for the 2023 repo...   
1  The runner runs quickly thru the running track...   
2  L'ensemble is better; I recieved the package i...   
3                  Their coming too sea if its reel.   
4  I love this movie! It's sweet, but with satiri...   

                                          lemmatized  
0                         [check, curl, mum, report]  
1  [runner, run, quickly, thru, running, track, g...  
2           [ensemble, better, received, package, u]  
3                                [coming, sea, reel]  
4             [love, movie, sweet, satirical, humor]  


---

## Summary of Results

| Step | Technique |
| :--- | :--- |
| **Noise** | RegEx for HTML/URLs, numbers, emojis |
| **Normalization** | Case folding |
| **Spelling** | Dictionary lookup (`pyspellchecker`) |
| **Tokenization** | Word tokenization |
| **Reduction** | Stemming vs. lemmatization |

**Follow-up:** You can extend this lab with **TF-IDF** or **word embeddings** on the cleaned tokens for a classification or downstream NLP task.